In [ ]:
# !pip install langgraph langchain langchain-openai python-dotenv

# https://docs.snowflake.com/LIMITEDACCESS/snowflake-cortex/open_ai_sdk
import os
os.environ["OPENAI_API_KEY"]='XXXXXXXXXXXXXXXXXXXX'
os.environ["OPENAI_API_BASE"] = 'https://SFSEEUROPE-EU_DEMO266.snowflakecomputing.com/api/v2/cortex/v1'

from openai import OpenAI

client = OpenAI(
  api_key=os.environ["OPENAI_API_KEY"],
  base_url=os.environ["OPENAI_API_BASE"]
)

response = client.chat.completions.create(
model="openai-gpt-5",
messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {
        "role": "user",
        "content": "How does a snowflake get its unique pattern?"
    }
  ]
)

print(response.choices[0].message)

from snowflake.snowpark import Session
connection_parameters = {
  "account": "SFSEEUROPE-EU_DEMO266",
  "user": "ahmed",
  "password": os.environ["OPENAI_API_KEY"],
  "role": 'ACCOUNTADMIN',  # optional
  "warehouse": "COMPUTE_WH",  # optional
  "database": "PROJECT_DB",  # optional
  "schema": "PUBLIC",  # optional
}

session = Session.builder.configs(connection_parameters).create()

# =========================
# Full working example:
# LangGraph support ticket agent
# - Priority evaluation
# - Cortex Search retrieval (Snowflake)
# - Response generation
# =========================

import os
from typing import TypedDict, List

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate

from langgraph.graph import StateGraph, START, END

from snowflake.snowpark.session import Session
from snowflake.core import Root


# -------------------------
# State definition
# -------------------------
class TicketAnalysisState(TypedDict, total=False):
    issue_description: str
    priority_level: str
    retrieved_context: List[str]
    final_response: str


# -------------------------
# LLM initialization
# -------------------------
reasoning_engine = ChatOpenAI(
    model="openai-gpt-5"
)


# -------------------------
# Node 1: Priority evaluation
# -------------------------
def evaluate_priority(state: TicketAnalysisState) -> dict:
    analysis_prompt = PromptTemplate(
        input_variables=["issue_description"],
        template="""
Evaluate this customer issue and assign a priority level using these criteria:

🔴 URGENT: Service outages, security vulnerabilities, payment failures, data loss
🟡 HIGH: Feature malfunctions affecting multiple users, integration failures
🟢 MEDIUM: Individual user problems, minor bugs, general inquiries
🔵 LOW: Feature requests, documentation questions, cosmetic issues

Customer Issue:
{issue_description}

Priority Assessment (URGENT/HIGH/MEDIUM/LOW):"""
    )

    msg = HumanMessage(
        content=analysis_prompt.format(
            issue_description=state["issue_description"]
        )
    )

    raw_priority = reasoning_engine.invoke([msg]).content.strip().upper()

    # Normalize output defensively
    if "URG" in raw_priority:
        priority = "URGENT"
    elif "HIGH" in raw_priority:
        priority = "HIGH"
    elif "MED" in raw_priority:
        priority = "MEDIUM"
    elif "LOW" in raw_priority:
        priority = "LOW"
    else:
        priority = "LOW"

    return {"priority_level": priority}




# -------------------------
# Cortex Search retrieval
# -------------------------
def retrieve(query: str) -> List[str]:
    root = Root(session)

    search_service = (
        root
        .databases["PROJECT_DB"]
        .schemas["PUBLIC"]
        .cortex_search_services["support_tickets_search_service"]
    )

    resp = search_service.search(
        query=query,
        columns=["body_answer"],
        limit=10
    )

    if resp.results:
        return [r["body_answer"] for r in resp.results]
    return []


# -------------------------
# Node 2: Response generation
# -------------------------
def generate_response(state: TicketAnalysisState) -> dict:
    issue = state["issue_description"]
    priority = state.get("priority_level", "UNKNOWN")

    try:
        context_chunks = retrieve(issue)
    except Exception:
        context_chunks = []

    context_text = (
        "\n\n---\n\n".join(context_chunks)
        if context_chunks
        else "NO_CONTEXT_FOUND"
    )

    response_prompt = PromptTemplate(
        input_variables=["issue_description", "priority_level", "context"],
        template="""
You are an intelligent customer support agent.

Ticket priority: {priority_level}

Customer issue:
{issue_description}

Relevant knowledge base excerpts:
{context}

Write a helpful support reply that:
- acknowledges the issue,
- gives concrete next steps,
- asks follow-up questions only if necessary,
- is concise and professional,
- includes escalation language if priority is URGENT or HIGH.

Final reply:"""
    )

    msg = HumanMessage(
        content=response_prompt.format(
            issue_description=issue,
            priority_level=priority,
            context=context_text,
        )
    )

    try:
        final_response = reasoning_engine.invoke([msg]).content
    except Exception as e:
        final_response = f"An internal error occurred while generating the response: {e}"

    return {
        "retrieved_context": context_chunks,
        "final_response": (final_response or "").strip(),
    }


# -------------------------
# Build LangGraph
# -------------------------
graph = StateGraph(state_schema=TicketAnalysisState)

graph.add_node("evaluate_priority", evaluate_priority)
graph.add_node("generate_response", generate_response)

graph.add_edge(START, "evaluate_priority")
graph.add_edge("evaluate_priority", "generate_response")
graph.add_edge("generate_response", END)

ticket_agent = graph.compile()


# -------------------------
# Example execution
# -------------------------
if __name__ == "__main__":
    initial_state: TicketAnalysisState = {
        "issue_description": "I’d like to see a dark mode option in the dashboard."
    }

    result = ticket_agent.invoke(initial_state)

    print("Priority:", result.get("priority_level"))
    print("\n--- Response ---\n")
    print(result.get("final_response"))
